In [15]:
# %% [markdown]
# # 📱 Google Play Review Fetcher (Project 1 - Phase 1)
# Fetches real customer feedback across 5 popular apps, maps ratings to sentiment, and exports a clean CSV.

# %% [code]
# Install dependencies (safe to run multiple times)
# !pip install -q google-play-scraper pandas tqdm

from google_play_scraper import reviews, Sort
import pandas as pd
from tqdm import tqdm
import time
import random
import os

# 🔧 Configuration
APP_LIST = [
    {"id": "com.jumia.android", "name": "Jumia (E-commerce)"},
    {"id": "com.glovo", "name": "Glovo (Food Delivery)"},
    {"id": "com.ubercab", "name": "Uber (Ride-hailing)"},
    {"id": "com.booking", "name": "Booking.com (Travel)"},
    {"id": "com.amazon.mShop.android.shopping", "name": "Amazon (Retail)"}
]

REVIEWS_PER_APP = 500
LANG = "en"
COUNTRY = "us"
SORT = Sort.MOST_RELEVANT

# 📁 Paths: Notebook is in notebooks/, data goes to root/data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")
OUTPUT_PATH = os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv")

# Create data folder if missing
os.makedirs(DATA_DIR, exist_ok=True)

# ⚡ Skip if already fetched (saves time & API calls)
if os.path.exists(OUTPUT_PATH):
    print(f"✅ Data already exists: {OUTPUT_PATH}")
    print("💡 Delete the file to re-fetch fresh reviews.")
else:
    all_reviews = []

    print("🔍 Fetching real customer reviews across 5 apps...\n")
    for app in tqdm(APP_LIST, desc="Apps", unit="app"):
        try:
            result, _ = reviews(
                app["id"], 
                lang=LANG, 
                country=COUNTRY, 
                sort=SORT, 
                count=REVIEWS_PER_APP
            )
            for r in result:
                r["app_name"] = app["name"]
                r["app_id"] = app["id"]
            all_reviews.extend(result)
            print(f"✅ {app['name']}: {len(result)} reviews fetched")
            time.sleep(random.uniform(1.5, 3.0))  # Respectful delay
        except Exception as e:
            print(f"⚠️ Failed to fetch {app['name']}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(all_reviews)
    if df.empty:
        raise RuntimeError("No reviews fetched. Check app IDs, network, or library version.")

    # 🏷️ Automatic Sentiment Labeling from Ratings
    def label_sentiment(score):
        if score >= 4.0: return "positive"
        elif score <= 2.0: return "negative"
        else: return "neutral"

    df["sentiment"] = df["score"].apply(label_sentiment)

    # 🧼 Basic Cleaning & Column Selection
    df_clean = df[["app_name", "userName", "content", "score", "sentiment", "at"]].copy()
    df_clean = df_clean.rename(columns={"content": "text", "at": "date"})
    df_clean = df_clean.dropna(subset=["text"])
    df_clean = df_clean[df_clean["text"].str.strip() != ""].reset_index(drop=True)

    # 💾 Export
    df_clean.to_csv(OUTPUT_PATH, index=False)

    print(f"\n✅ Saved {len(df_clean)} reviews to {OUTPUT_PATH}")
    
# 📊 Show stats (works whether fetched or loaded)
df_stats = pd.read_csv(OUTPUT_PATH) if not os.path.exists(OUTPUT_PATH) else pd.read_csv(OUTPUT_PATH)
print("📊 Sentiment Distribution:")
print(df_stats["sentiment"].value_counts())
print("\n📈 App-wise Distribution:")
print(df_stats["app_name"].value_counts())

🔍 Fetching real customer reviews across 5 apps...



Apps:   0%|          | 0/5 [00:00<?, ?app/s]

✅ Jumia (E-commerce): 500 reviews fetched


Apps:  20%|██        | 1/5 [00:03<00:12,  3.19s/app]

✅ Glovo (Food Delivery): 500 reviews fetched


...

✅ Booking.com (Travel): 500 reviews fetched


Apps:  80%|████████  | 4/5 [00:12<00:03,  3.26s/app]

✅ Amazon (Retail): 500 reviews fetched


Apps: 100%|██████████| 5/5 [00:16<00:00,  3.31s/app]


✅ Saved 2500 reviews to c:\Users\ija\OneDrive - WEVIOO\Documents\sentiment analysis\notebooks\..\data\gplay_multi_app_reviews.csv
📊 Sentiment Distribution:
sentiment
negative    1812
positive     446
neutral      242
Name: count, dtype: int64

📈 App-wise Distribution:
app_name
Jumia (E-commerce)       500
Glovo (Food Delivery)    500
Uber (Ride-hailing)      500
Booking.com (Travel)     500
Amazon (Retail)          500
Name: count, dtype: int64


In [16]:
#!pip install pandas numpy scikit-learn spacy tqdm matplotlib
#!pip install google_play_scraper


In [18]:
import pandas as pd
import os

# Fix: Notebook is in notebooks/, so go up one level to reach data/
DATA_DIR = os.path.join(os.getcwd(), "..", "data")

# Load ONLY 5,000 from Yelp (fast, sufficient for DistilBERT)
yelp_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), 
                      header=None, 
                      names=["label", "text"], 
                      nrows=5000)

yelp_df["label"] = yelp_df["label"].map({1: "negative", 2: "positive"})
yelp_df["source"] = "yelp_public"

# Load your Google Play data (2,500 reviews)
gplay_df = pd.read_csv(os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv"))
gplay_df = gplay_df.rename(columns={"sentiment": "label", "content": "text"})
gplay_df["source"] = "google_play"

# Merge
df = pd.concat([yelp_df, gplay_df], ignore_index=True)
df = df.rename(columns={"label": "sentiment", "text": "review"})

print(f"Total: {len(df)} reviews")
print(df["sentiment"].value_counts())
print(f"Sources: {df['source'].value_counts().to_dict()}")

Total: 7500 reviews
sentiment
negative    4424
positive    2834
neutral      242
Name: count, dtype: int64
Sources: {'yelp_public': 5000, 'google_play': 2500}
